# 🚗 Driver Drowsiness Detection System  
### Computer Vision Project | Real-Time Driver Monitoring using EAR, MAR & Head Pose

---

## 📌 How it works:

- 📷 Uses your laptop camera for real-time monitoring  
- 👤 Detects face and tracks 68 facial landmarks (dlib)  
- 👁️ Calculates Eye Aspect Ratio (EAR) to detect eye closure  
- 👄 Uses Mouth Aspect Ratio (MAR) to detect yawning  
- 🧠 Uses Head Pose Estimation to detect head tilting/down posture  
- 🚨 If drowsiness signs persist (eyes closed / yawning / head down), an alert is triggered  

---

## 🧠 Detection Signals Used:

- ✔️ EAR → Eye closure detection  
- ✔️ MAR → Yawning detection  
- ✔️ Head Pose → Sleepy head movement detection  

---

## ⌨️ Controls:

- ▶️ Run cells from top to bottom  
- ⌨️ Press `Q` in the camera window to stop the system  

## 📥 Cell 2 — Download Face Landmark Model
Downloads the **dlib 68-point shape predictor** model automatically.

In [3]:
import os
import urllib.request
import bz2

MODEL_FILE = "shape_predictor_68_face_landmarks.dat"
MODEL_URL  = "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
BZ2_FILE   = MODEL_FILE + ".bz2"

if os.path.exists(MODEL_FILE):
    print(f"✅ Model already exists: {MODEL_FILE}")
else:
    print("📥 Downloading shape predictor model (~99MB)...")
    print("   Please wait, this may take a minute...")

    def show_progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        pct = min(100, downloaded * 100 / total_size)
        if block_num % 500 == 0:
            print(f"   {pct:.1f}% downloaded...", end="\r")

    urllib.request.urlretrieve(MODEL_URL, BZ2_FILE, show_progress)
    print("\n   Download complete! Extracting...")

    with bz2.BZ2File(BZ2_FILE) as f_in:
        data = f_in.read()
    with open(MODEL_FILE, "wb") as f_out:
        f_out.write(data)

    os.remove(BZ2_FILE)
    print(f"✅ Model ready: {MODEL_FILE}")

✅ Model already exists: shape_predictor_68_face_landmarks.dat


## ⚙️ Cell 3 — Configuration Settings
Adjust these values to tune sensitivity.

In [4]:
# ─────────────────────────────────────────────────────────
# CONFIGURATION — Tune these to match your needs
# ─────────────────────────────────────────────────────────

EAR_THRESHOLD     = 0.25
EAR_CONSEC_FRAMES = 20

CAMERA_INDEX      = 0

ALERT_SOUND_FILE  = None


# ================= NEW ADDITIONS =================

# ✅ Yawn Detection (MAR)
MAR_THRESHOLD     = 0.6    # Higher = mouth open (yawning)
MAR_CONSEC_FRAMES = 15     # Frames for yawning detection

# ✅ Head Pose Detection
HEAD_POSE_THRESHOLD = 0.5  # Higher = head tilt/down detection

# =================================================

print("✅ Configuration loaded!")
print(f"   EAR Threshold     : {EAR_THRESHOLD}")
print(f"   Consecutive Frames: {EAR_CONSEC_FRAMES}")
print(f"   Camera Index      : {CAMERA_INDEX}")

# NEW prints (optional but helpful)
print(f"   MAR Threshold     : {MAR_THRESHOLD}")
print(f"   Head Pose Thresh  : {HEAD_POSE_THRESHOLD}")

✅ Configuration loaded!
   EAR Threshold     : 0.25
   Consecutive Frames: 20
   Camera Index      : 0
   MAR Threshold     : 0.6
   Head Pose Thresh  : 0.5


## 🧠 Cell 4 — Load Libraries & Helper Functions

In [5]:
import cv2
import dlib
import numpy as np
import pygame
import time
import threading
from imutils import face_utils
from scipy.spatial import distance

# ─────────────────────────────────────────────────────────
# FUNCTION: Eye Aspect Ratio
# ─────────────────────────────────────────────────────────
def eye_aspect_ratio(eye):
    """
    Computes Eye Aspect Ratio (EAR).

    EAR = (||p2-p6|| + ||p3-p5||) / (2 * ||p1-p4||)

    Open eyes  → EAR ≈ 0.30+
    Closed eyes → EAR ≈ 0.20 or less
    """
    A = distance.euclidean(eye[1], eye[5])  # vertical
    B = distance.euclidean(eye[2], eye[4])  # vertical
    C = distance.euclidean(eye[0], eye[3])  # horizontal
    return (A + B) / (2.0 * C)


# ─────────────────────────────────────────────────────────
# FUNCTION: Generate Beep Alert Sound
# ─────────────────────────────────────────────────────────
def generate_beep_sound(frequency=1000, duration=0.6, volume=0.9):
    """Generates a beep tone using numpy and pygame."""
    sample_rate = 44100
    n_samples   = int(sample_rate * duration)
    t           = np.linspace(0, duration, n_samples, False)

    # Sine wave
    wave = np.sin(2 * np.pi * frequency * t)

    # Apply fade in/out to avoid click noise
    fade = int(sample_rate * 0.05)
    wave[:fade]  *= np.linspace(0, 1, fade)
    wave[-fade:] *= np.linspace(1, 0, fade)

    wave = (wave * volume * 32767).astype(np.int16)
    wave = np.column_stack([wave, wave])  # stereo
    return pygame.sndarray.make_sound(wave)


# ─────────────────────────────────────────────────────────
# FUNCTION: Play Alert (non-blocking)
# ─────────────────────────────────────────────────────────
alert_sound = None

def init_sound():
    """Initialize pygame mixer and load/generate alert sound."""
    global alert_sound
    try:
        pygame.mixer.init(frequency=44100, size=-16, channels=2, buffer=512)
        if ALERT_SOUND_FILE and os.path.exists(ALERT_SOUND_FILE):
            alert_sound = pygame.mixer.Sound(ALERT_SOUND_FILE)
        else:
            alert_sound = generate_beep_sound(frequency=1000, duration=0.6)
        print("✅ Sound system initialized!")
    except Exception as e:
        print(f"⚠️  Sound init failed: {e} — alert will be visual only.")


def play_alert_async():
    """Play alert sound in a separate thread so video doesn't freeze."""
    def _play():
        try:
            if alert_sound and not pygame.mixer.get_busy():
                alert_sound.play()
        except:
            pass
    threading.Thread(target=_play, daemon=True).start()


# ─────────────────────────────────────────────────────────
# FUNCTION: Draw EAR Bar
# ─────────────────────────────────────────────────────────
def draw_ear_bar(frame, ear, threshold, x=10, y=100, w=200, h=20):
    """Draws a visual EAR meter bar on the frame."""
    # Background bar
    cv2.rectangle(frame, (x, y), (x + w, y + h), (50, 50, 50), -1)

    # Filled portion — clamp EAR to 0–0.4 range
    ratio     = min(ear / 0.4, 1.0)
    fill_w    = int(w * ratio)
    bar_color = (0, 200, 0) if ear >= threshold else (0, 0, 255)
    cv2.rectangle(frame, (x, y), (x + fill_w, y + h), bar_color, -1)

    # Border
    cv2.rectangle(frame, (x, y), (x + w, y + h), (200, 200, 200), 1)

    # Threshold marker
    thresh_x = int(x + w * (threshold / 0.4))
    cv2.line(frame, (thresh_x, y - 3), (thresh_x, y + h + 3), (0, 255, 255), 2)

    cv2.putText(frame, "EAR Meter", (x, y - 6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)


print("✅ Helper functions ready!")
# ─────────────────────────────────────────────────────────
# FUNCTION: Mouth Aspect Ratio (Yawn Detection)
# ─────────────────────────────────────────────────────────
def mouth_aspect_ratio(mouth):
    """
    Computes Mouth Aspect Ratio (MAR)

    High MAR → mouth open → yawning
    """
    A = distance.euclidean(mouth[2], mouth[10])  # vertical
    B = distance.euclidean(mouth[4], mouth[8])   # vertical
    C = distance.euclidean(mouth[0], mouth[6])   # horizontal
    return (A + B) / (2.0 * C)


# ─────────────────────────────────────────────────────────
# FUNCTION: Head Pose Detection
# ─────────────────────────────────────────────────────────
def get_head_pose(shape, frame):
    """
    Estimate head pose using facial landmarks
    Returns rotation vector
    """
    image_points = np.array([
        shape[30],  # Nose tip
        shape[8],   # Chin
        shape[36],  # Left eye
        shape[45],  # Right eye
        shape[48],  # Left mouth
        shape[54]   # Right mouth
    ], dtype="double")

    model_points = np.array([
        (0.0, 0.0, 0.0),
        (0.0, -330.0, -65.0),
        (-225.0, 170.0, -135.0),
        (225.0, 170.0, -135.0),
        (-150.0, -150.0, -125.0),
        (150.0, -150.0, -125.0)
    ])

    focal_length = frame.shape[1]
    center = (frame.shape[1] / 2, frame.shape[0] / 2)

    camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]
    ], dtype="double")

    dist_coeffs = np.zeros((4, 1))

    success, rotation_vector, translation_vector = cv2.solvePnP(
        model_points, image_points, camera_matrix, dist_coeffs
    )

    return rotation_vector


print("✅ MAR + Head Pose functions added!")

C:\Users\star\anaconda3\envs\torch_ok\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.10.19)
Hello from the pygame community. https://www.pygame.org/contribute.html
✅ Helper functions ready!
✅ MAR + Head Pose functions added!


## 🤖 Cell 5 — Load Detection Models

In [6]:
MODEL_FILE = "shape_predictor_68_face_landmarks.dat"

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(
        f"❌ Model file not found: {MODEL_FILE}\n"
        "Please run Cell 2 first to download the model."
    )

print("[INFO] Loading dlib face detector...")
detector = dlib.get_frontal_face_detector()

print("[INFO] Loading 68-point shape predictor...")
predictor = dlib.shape_predictor(MODEL_FILE)

# Get eye landmark index ranges from dlib
(lStart, lEnd) = face_utils.FACIAL_LANDMARKS_IDXS["left_eye"]
(rStart, rEnd) = face_utils.FACIAL_LANDMARKS_IDXS["right_eye"]

# ================= NEW ADDITIONS =================

# ✅ Mouth landmarks (for Yawn Detection - MAR)
(mStart, mEnd) = (48, 68)

# =================================================

# Init sound
init_sound()

print("\n✅ All models loaded and ready!")
print(f"   Left eye  landmarks : {lStart} → {lEnd}")
print(f"   Right eye landmarks : {rStart} → {rEnd}")

# Optional print (debug)
print(f"   Mouth landmarks     : {mStart} → {mEnd}")

[INFO] Loading dlib face detector...
[INFO] Loading 68-point shape predictor...
✅ Sound system initialized!

✅ All models loaded and ready!
   Left eye  landmarks : 42 → 48
   Right eye landmarks : 36 → 42
   Mouth landmarks     : 48 → 68


## 🎥 Cell 6 — Run Real-Time Drowsiness Detection
**This opens your camera. Press `Q` in the video window to stop.**

In [7]:
# ─────────────────────────────────────────────────────────
# MAIN DETECTION LOOP (FINAL FIXED)
# ─────────────────────────────────────────────────────────

cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print("❌ Cannot open camera. Check CAMERA_INDEX in Cell 3.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    print("✅ Camera opened successfully!")
    print("📷 Real-time detection running...")
    print("   Press Q in the video window to stop.\n")

    # ── State Variables ───────────────────────────────────
    frame_counter  = 0
    total_alerts   = 0
    alert_active   = False
    start_time     = time.time()
    fps_counter    = 0
    fps            = 0

    # ── NEW ADDITIONS ─────────────────────────────────────
    mar_counter = 0
    yawn_alert  = False
    head_alert  = False

    y_true = []
    y_pred = []

    # ── Main Loop ─────────────────────────────────────────
    while True:
        ret, frame = cap.read()
        if not ret:
            print("❌ Failed to read frame from camera.")
            break

        frame = cv2.flip(frame, 1)
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        fps_counter += 1
        elapsed = time.time() - start_time
        if elapsed >= 1.0:
            fps         = fps_counter
            fps_counter = 0
            start_time  = time.time()

        faces     = detector(gray, 0)
        ear       = 0.0
        mar       = 0.0
        face_found = len(faces) > 0

        for face in faces:
            shape = predictor(gray, face)
            shape = face_utils.shape_to_np(shape)

            # Eyes
            leftEye  = shape[lStart:lEnd]
            rightEye = shape[rStart:rEnd]

            leftEAR  = eye_aspect_ratio(leftEye)
            rightEAR = eye_aspect_ratio(rightEye)
            ear      = (leftEAR + rightEAR) / 2.0

            # ── MAR (Yawn)
            mouth = shape[mStart:mEnd]
            mar   = mouth_aspect_ratio(mouth)

            if mar > MAR_THRESHOLD:
                mar_counter += 1
                if mar_counter >= MAR_CONSEC_FRAMES:
                    yawn_alert = True
            else:
                mar_counter = 0
                yawn_alert = False

            # ── HEAD POSE (FIXED)
            pitch = get_head_pose(shape, frame)

            # ensure scalar value
            if isinstance(pitch, (list, np.ndarray)):
                pitch_val = float(pitch[0])
            else:
                pitch_val = float(pitch)

            head_alert = pitch_val > HEAD_POSE_THRESHOLD

            # ── Draw
            eye_color = (0, 255, 0) if ear >= EAR_THRESHOLD else (0, 100, 255)
            cv2.drawContours(frame, [cv2.convexHull(leftEye)],  -1, eye_color, 1)
            cv2.drawContours(frame, [cv2.convexHull(rightEye)], -1, eye_color, 1)
            cv2.drawContours(frame, [cv2.convexHull(mouth)], -1, (255, 0, 0), 1)

            x1, y1 = face.left(), face.top()
            x2, y2 = face.right(), face.bottom()
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

            # ── COMBINED LOGIC (FIXED)
            if (ear < EAR_THRESHOLD) or yawn_alert or head_alert:
                frame_counter += 1

                if frame_counter >= EAR_CONSEC_FRAMES:
                    if not alert_active:
                        total_alerts += 1
                        alert_active  = True

                    play_alert_async()

                    overlay = frame.copy()
                    cv2.rectangle(overlay, (0, 0), (frame.shape[1], 90), (0, 0, 180), -1)
                    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

                    cv2.putText(frame, "DROWSINESS DETECTED!",
                                (10, 40), cv2.FONT_HERSHEY_DUPLEX,
                                1.1, (255, 255, 255), 2)
                else:
                    remaining = EAR_CONSEC_FRAMES - frame_counter
                    cv2.putText(frame, f"Eyes closing... ({remaining})",
                                (10, 35), cv2.FONT_HERSHEY_SIMPLEX,
                                0.7, (0, 140, 255), 2)
            else:
                if frame_counter > 0:
                    frame_counter = max(0, frame_counter - 2)
                else:
                    alert_active = False

            # Extra labels
            if yawn_alert:
                cv2.putText(frame, "YAWNING!", (10, 120),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

            if head_alert:
                cv2.putText(frame, "HEAD DOWN!", (10, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

        # ── HUD
        panel_y = frame.shape[0] - 60
        panel   = frame.copy()
        cv2.rectangle(panel, (0, panel_y), (frame.shape[1], frame.shape[0]), (20, 20, 20), -1)
        cv2.addWeighted(panel, 0.7, frame, 0.3, 0, frame)

        status_text  = "NO FACE DETECTED" if not face_found else ("STATUS: DROWSY" if alert_active else "STATUS: AWAKE")
        status_color = (0,165,255) if not face_found else ((0,0,255) if alert_active else (0,220,0))

        cv2.putText(frame, status_text,
                    (10, panel_y + 22), cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, status_color, 2)

        cv2.putText(frame,
                    f"EAR: {ear:.2f} | MAR: {mar:.2f} | Alerts: {total_alerts} | FPS: {fps}",
                    (10, panel_y + 48),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

        if face_found:
            draw_ear_bar(frame, ear, EAR_THRESHOLD, x=frame.shape[1]-220, y=10)

        cv2.imshow("Driver Drowsiness Detection", frame)

        # Metrics
        pred = 1 if alert_active else 0
        true = pred
        y_true.append(true)
        y_pred.append(pred)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n[INFO] Quit signal received.")
            break

    cap.release()
    cv2.destroyAllWindows()
    try:
        pygame.mixer.quit()
    except:
        pass

    print("\n📊 Session Summary")
    print(f"   Total Alerts : {total_alerts}")
    print("✅ Program ended.")

✅ Camera opened successfully!
📷 Real-time detection running...
   Press Q in the video window to stop.



C:\Users\star\AppData\Local\Temp\ipykernel_20584\3170928615.py:84: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pitch_val = float(pitch[0])



[INFO] Quit signal received.

📊 Session Summary
   Total Alerts : 2
✅ Program ended.
